<a href="https://colab.research.google.com/github/SpikemenDan/vawt-research/blob/main/Aarion's_Modified_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
#@title Block 1: Master Initialization & Historic Data {display-mode: "form"}
# ==============================================================================
import os
import numpy as np
import pandas as pd
from google.colab import drive

# Smart Drive Mount
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("✅ Google Drive already mounted.")

MASTER_DIR = '/content/drive/MyDrive/TurbineResearch'
os.makedirs(MASTER_DIR, exist_ok=True)


def save_and_track_matrix(df, file_name, directory=MASTER_DIR):
    if not file_name.endswith('.parquet'):
        file_name += '.parquet'
    full_path = os.path.join(directory, file_name)
    df.to_parquet(full_path, engine='auto', index=False)
    var_name = file_name.replace('.parquet', '').replace(' ', '_').lower()
    print(f"SUCCESS: {var_name} saved to Drive.")


# Track BOTH peaks and raw sweeps to prevent data loss
historic_df_list = []
historic_raw_list = []

# ==============================================================================
# 1. LOCALIZED PEAK EXTRACTION: HUNT 2024 DATASET
# ==============================================================================
RE_TARGETS = np.array([0.75e5, 1.6e5, 2.7e5, 8.3e5])
RE_LABELS = {t: f"{t:.1e}" for t in RE_TARGETS}


def snap_to_nearest(values, targets=RE_TARGETS):
    """Vectorized: snap every value to its closest target (ties -> first target)."""
    v = np.asarray(values, dtype=float)
    return targets[np.abs(v[:, None] - targets[None, :]).argmin(axis=1)]


hunt_parquet_path = os.path.join(MASTER_DIR, 'hunt_kinematic_master.parquet')

if os.path.exists(hunt_parquet_path):
    print("Loading processed Hunt .parquet file...")
    df_hunt_raw = pd.read_parquet(hunt_parquet_path)

    if 'dataset_source' not in df_hunt_raw.columns:
        df_hunt_raw['dataset_source'] = 'Hunt 2024'

    df_hunt_raw['Re_D_binned'] = snap_to_nearest(df_hunt_raw['Re_D'])

    # STRICT FIX: Explicitly include alpha_p to prevent sawtooth grouping
    df_hunt_raw['label'] = ("Hunt: N=" + df_hunt_raw['N'].astype(int).astype(str)
                            + ", c/R=" + df_hunt_raw['cToR'].round(2).astype(str)
                            + ", Re=" + df_hunt_raw['Re_D_binned'].map(RE_LABELS)
                            + ", alpha_p=" + df_hunt_raw['alpha_p'].astype(str))

    if 'Dynamic_Flow_Opacity' not in df_hunt_raw.columns and 'solidity' in df_hunt_raw.columns:
        df_hunt_raw['Dynamic_Flow_Opacity'] = df_hunt_raw['solidity'] * df_hunt_raw['TSR']

    hunt_group_cols = ['N', 'cToR', 'alpha_p', 'Re_D_binned']
    df_hunt_peaks = df_hunt_raw.loc[df_hunt_raw.groupby(hunt_group_cols)['Cp'].idxmax()].copy()

    historic_df_list.append(df_hunt_peaks)
    historic_raw_list.append(df_hunt_raw)

    print("✅ Ingested Hunt dataset.")
    print(f"   - Raw sweep points added: {len(df_hunt_raw)}")
    print(f"   - Localized absolute peaks: {len(df_hunt_peaks)} based on {hunt_group_cols}.")
else:
    print(f"⚠️ Hunt dataset not found at {hunt_parquet_path}.")

# ==============================================================================
# 2. LOCALIZED PEAK EXTRACTION: UNH-RVAT DATASET
# ==============================================================================
unh_parquet_path = os.path.join(MASTER_DIR, 'unh_kinematic_master.parquet')

if os.path.exists(unh_parquet_path):
    print("\nLoading processed UNH-RVAT .parquet file...")
    df_unh_raw = pd.read_parquet(unh_parquet_path)

    if 'dataset_source' not in df_unh_raw.columns:
        df_unh_raw['dataset_source'] = 'UNH-RVAT'

    df_unh_raw = df_unh_raw.rename(columns={'mean_tsr': 'TSR', 'mean_cp': 'Cp'})

    if 'cToR' not in df_unh_raw.columns:
        df_unh_raw['cToR'] = 0.14 / 0.5
    if 'N' not in df_unh_raw.columns:
        df_unh_raw['N'] = 3

    if 'solidity' not in df_unh_raw.columns:
        df_unh_raw['solidity'] = (df_unh_raw['N'] * df_unh_raw['cToR']) / (2 * np.pi)
    if 'Dynamic_Flow_Opacity' not in df_unh_raw.columns:
        df_unh_raw['Dynamic_Flow_Opacity'] = df_unh_raw['solidity'] * df_unh_raw['TSR']

    # Explicit root label so each tow-speed sweep stays its own curve.
    # (Fixed: the old code crashed here if 'mean_tow_speed' was missing.)
    if 'mean_tow_speed' in df_unh_raw.columns:
        df_unh_raw['tow_speed_binned'] = df_unh_raw['mean_tow_speed'].round(1)
        unh_group_cols = ['tow_speed_binned']
        df_unh_raw['label'] = "UNH-RVAT: U=" + df_unh_raw['tow_speed_binned'].astype(str) + " m/s"
    else:
        unh_group_cols = ['source_file'] if 'source_file' in df_unh_raw.columns else ['dataset_source']
        df_unh_raw['label'] = "UNH-RVAT: " + df_unh_raw[unh_group_cols[0]].astype(str)

    df_unh_peaks = df_unh_raw.loc[df_unh_raw.groupby(unh_group_cols)['Cp'].idxmax()].copy()

    historic_df_list.append(df_unh_peaks)
    historic_raw_list.append(df_unh_raw)

    print("✅ Ingested UNH-RVAT dataset.")
    print(f"   - Raw sweep points added: {len(df_unh_raw)}")
    print(f"   - Localized absolute peaks: {len(df_unh_peaks)} based on {unh_group_cols}.")
else:
    print(f"⚠️ UNH dataset not found at {unh_parquet_path}.")

✅ Google Drive already mounted.
Loading processed Hunt .parquet file...
✅ Ingested Hunt dataset.
   - Raw sweep points added: 4083
   - Localized absolute peaks: 223 based on ['N', 'cToR', 'alpha_p', 'Re_D_binned'].

Loading processed UNH-RVAT .parquet file...
✅ Ingested UNH-RVAT dataset.
   - Raw sweep points added: 125
   - Localized absolute peaks: 5 based on ['tow_speed_binned'].


In [41]:
#@title Block 2: Paper Grouping + Legend Panel {display-mode: "form"}
# ==============================================================================
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors
import ipywidgets as widgets

# ------------------------------------------------------------------------------
# MANUAL MERGES (optional)
# If two names still show up as separate papers, map them here.
# Matching ignores capitalization and spacing.
#   PAPER_ALIASES = {"Howell 32": "Howell et al. [32]", "howell et al 32": "Howell et al. [32]"}
# ------------------------------------------------------------------------------
PAPER_ALIASES = {}

# ------------------------------------------------------------------------------
# COLUMN RULES (shared by Block 3, the legend, and the Dataset Manager)
#   Every uploaded CSV must have:  TSR, Cp, cToR, Re_D   (header "Re" is accepted too)
#   N is optional (needed only for solidity / flow opacity and the by-blade-count phases).
#   ANY other column is treated as an optional condition: it is kept, it splits curves
#   that differ in it, and it appears in the curve name. Nothing is hard-coded per column.
# ------------------------------------------------------------------------------
REQUIRED_COLS = ["TSR", "Cp", "cToR", "Re_D"]
NON_CONDITION_COLS = {
    "TSR", "Cp", "solidity", "Dynamic_Flow_Opacity", "Dynamic_Blockage",
    "dataset_source", "original_dataset_source", "source_file", "ingest_file_key",
    "label", "curve_name", "config_id", "filename", "Action", "Notes",
    "Re_basis", "Pitch_basis", "Pitch_definition",
}


def condition_columns(columns):
    """Columns that describe HOW a curve was measured (everything that isn't data/bookkeeping)."""
    return [c for c in columns
            if c not in NON_CONDITION_COLS and not str(c).startswith(("Unnamed:", "assumed_"))]


_TRAILING_SWEEP_TAG = re.compile(
    r"[\s,;:_\-–]*\b(?:fig(?:ure)?\.?\s*\d+\w*|[A-Za-z_][\w/\-]*\s*=.*)$",    # fig. N, or any trailing "name = value" tag
    re.IGNORECASE,
)


def _norm_key(text):
    return re.sub(r"\s+", " ", str(text).replace("\xa0", " ")).strip().lower()


def _clean_paper_name(name):
    """'Howell et al.\xa0[32] (N=3)' -> 'Howell et al. [32]'"""
    s = str(name).replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = s.split("(")[0].strip()
    s = _TRAILING_SWEEP_TAG.sub("", s).strip(" ,;:_-–")
    return s or "Unknown source"


def prepare_frames(frames, verbose=True):
    """
    Shared cleanup used by BOTH action cells so they always agree.
      - merges messy dataset names into one name per paper
      - guarantees N / cToR / Re_D / opacity columns exist (N may be NaN)
      - gives every curve a unique, readable `label` (and `curve_name` for the legend panel)
    Pass a list of dataframes (e.g. [df_raw, df_peaks]); returns the cleaned list.
    """
    frames = [f.copy() for f in frames]
    aliases = {_norm_key(k): v for k, v in PAPER_ALIASES.items()}

    # ---- 1. one canonical name per paper ----
    raw_names = pd.unique(pd.concat([f["dataset_source"] for f in frames]).dropna().astype(str))
    resolved = {}
    for raw in raw_names:
        cleaned = _clean_paper_name(raw)
        resolved[raw] = aliases.get(_norm_key(raw)) or aliases.get(cleaned.lower()) or cleaned

    canon = {}
    for name in sorted(set(resolved.values())):
        canon.setdefault(name.lower(), name)
    final = {raw: canon[name.lower()] for raw, name in resolved.items()}

    id_cols = ["N", "cToR", "Re_D"]          # always present as numeric (N may be NaN)
    seen = dict.fromkeys(c for f in frames for c in f.columns)
    extra_cols = [c for c in condition_columns(seen) if c not in id_cols]   # any optional condition
    for f in frames:
        f["original_dataset_source"] = f["dataset_source"]
        f["dataset_source"] = f["dataset_source"].astype(str).map(final).fillna("Unknown source")

        if "Dynamic_Flow_Opacity" not in f.columns and "Dynamic_Blockage" in f.columns:
            f["Dynamic_Flow_Opacity"] = f["Dynamic_Blockage"]
        elif "Dynamic_Flow_Opacity" in f.columns and "Dynamic_Blockage" not in f.columns:
            f["Dynamic_Blockage"] = f["Dynamic_Flow_Opacity"]

        for col in id_cols:
            if col not in f.columns:
                f[col] = np.nan
            f[col] = pd.to_numeric(f[col], errors="coerce")

        # identity of a curve: config_id for uploaded data, existing label for Hunt / UNH
        if "label" not in f.columns:
            f["label"] = np.nan
        f["label"] = f["label"].astype(object)
        if "config_id" in f.columns:
            has_id = f["config_id"].notna()
            f.loc[has_id, "label"] = f.loc[has_id, "config_id"].astype(str)
        missing = f["label"].isna()
        if missing.any():
            sub = f.loc[missing, id_cols].astype(str)
            f.loc[missing, "label"] = "params:" + sub["N"].str.cat([sub["cToR"], sub["Re_D"]], sep="|")
        f["_orig"] = f["label"].astype(str)

    # ---- 2. readable + unique label per curve ----
    keep = ["dataset_source", "_orig"] + id_cols + extra_cols
    uniq = pd.concat([f.reindex(columns=keep) for f in frames]).drop_duplicates(["dataset_source", "_orig"])

    def describe(r):
        orig = str(r["_orig"])
        body = orig.split(": ", 1)[1] if ": " in orig else orig
        if ": " in orig and "=" in body:          # already readable (Hunt / UNH style)
            return body
        parts = []
        if pd.notna(r["N"]):       parts.append(f"N={int(r['N'])}")
        if pd.notna(r["cToR"]):    parts.append(f"c/R={r['cToR']:.4g}")
        if pd.notna(r["Re_D"]):    parts.append(f"Re={r['Re_D']:.1e}")
        for c in extra_cols:                       # whatever optional conditions this curve has
            v = r[c]
            if pd.notna(v):
                parts.append(f"{c}={v:g}" if isinstance(v, (int, float, np.number)) and not isinstance(v, bool) else f"{c}={v}")
        return ", ".join(parts) if parts else body

    uniq = uniq.sort_values(["dataset_source", "_orig"]).reset_index(drop=True)
    uniq["_name"] = uniq.apply(describe, axis=1)
    grp = uniq.groupby(["dataset_source", "_name"])["_orig"]
    dup_size = grp.transform("size")
    dup_rank = grp.cumcount() + 1
    uniq["_name"] = np.where(dup_size > 1, uniq["_name"] + " #" + dup_rank.astype(str), uniq["_name"])
    uniq["_label"] = uniq["dataset_source"] + ": " + uniq["_name"]

    # One vectorized lookup per frame instead of a Python loop over every row.
    lookup = uniq.set_index(["dataset_source", "_orig"])[["_label", "_name"]]
    for f in frames:
        got = lookup.reindex(pd.MultiIndex.from_arrays([f["dataset_source"], f["_orig"]]))
        f["label"] = got["_label"].to_numpy()
        f["curve_name"] = got["_name"].to_numpy()
        f.drop(columns=["_orig"], inplace=True)

    if verbose:
        groups = {}
        for raw, paper in final.items():
            groups.setdefault(paper, []).append(raw)
        print(f"📚 {len(groups)} papers detected (from {len(final)} raw dataset names).")
        for paper in sorted(groups):
            raws = sorted(set(groups[paper]))
            if len(raws) > 1:
                shown = "; ".join(repr(r) for r in raws[:4]) + (" …" if len(raws) > 4 else "")
                print(f"   • {paper}  ←  merged {len(raws)} names: {shown}")

    return frames


_MARKER_OPTIONS = ["o", "X", "s", "^", "D", "v", "P", "*"]
_MARKER_GLYPH = {"o": "●", "X": "✖", "s": "■", "^": "▲", "D": "◆", "v": "▼", "P": "✚", "*": "★"}


def make_style_dicts(papers):
    papers = list(papers)
    pal = sns.color_palette("tab10" if len(papers) <= 10 else "tab20", max(len(papers), 1))
    palette_dict = {p: pal[i % len(pal)] for i, p in enumerate(papers)}
    marker_dict = {p: _MARKER_OPTIONS[i % len(_MARKER_OPTIONS)] for i, p in enumerate(papers)}
    return palette_dict, marker_dict


# Every row/box inside the legend panel is told NOT to shrink. Without this, the
# panel (a fixed-height flex column) squashes rows when two papers are expanded at
# once and their contents spill over each other. With it, the panel just scrolls.
_FIXED = dict(flex="0 0 auto")


class LegendSelector:
    """
    Side panel that doubles as a legend + filter.
      - ONE checkbox per paper: turns every curve of that paper on/off
      - click ▸ to expand a paper and isolate individual curves
      - 'solo' = show only that paper; 'Show all' / 'Hide all' on top
    """

    def __init__(self, frames, palette_dict, marker_dict):
        curves = pd.concat(
            [f[["dataset_source", "label", "curve_name", "N", "cToR", "Re_D"]] for f in frames]
        ).drop_duplicates(["dataset_source", "label"])

        self.papers = sorted(curves["dataset_source"].unique())
        self._callbacks = []
        self._busy = False
        self.master = {}
        self.curve_cbs = {}
        self._counts = {}
        self._bodies = {}

        rows = []
        for paper in self.papers:
            sub = curves[curves["dataset_source"] == paper].sort_values(
                ["N", "cToR", "Re_D", "label"], na_position="last"
            )
            color = mcolors.to_hex(palette_dict.get(paper, (0.5, 0.5, 0.5)))
            glyph = _MARKER_GLYPH.get(marker_dict.get(paper, "o"), "●")
            swatch = widgets.HTML(
                f"<span style='color:{color};font-size:17px;line-height:26px'>{glyph}</span>",
                layout=widgets.Layout(width="22px", **_FIXED),
            )

            cbs = {}
            for lab, name in zip(sub["label"], sub["curve_name"]):
                cbs[lab] = widgets.Checkbox(
                    value=True, description=name, indent=False,
                    layout=widgets.Layout(width="max-content", **_FIXED),
                )
            self.curve_cbs[paper] = cbs
            self._counts[paper] = len(cbs)

            master = widgets.Checkbox(
                value=True, indent=False,
                style={"description_width": "initial"},
                layout=widgets.Layout(width="auto", flex="1 1 auto"),
            )
            self.master[paper] = master

            expand = widgets.ToggleButton(
                value=False, description="▸", tooltip="Show / hide individual curves",
                layout=widgets.Layout(width="34px", **_FIXED),
            )
            solo = widgets.Button(
                description="solo", tooltip="Show only this paper",
                layout=widgets.Layout(width="52px", **_FIXED),
            )
            body = widgets.VBox(
                list(cbs.values()),
                layout=widgets.Layout(display="none", margin="0 0 6px 34px",
                                      max_height="240px", overflow="auto", **_FIXED),
            )
            self._bodies[paper] = body

            self._wire(paper, master, expand, solo, body)
            self._refresh_label(paper)
            rows.append(widgets.VBox(
                [widgets.HBox([swatch, master, expand, solo], layout=widgets.Layout(**_FIXED)), body],
                layout=widgets.Layout(**_FIXED),
            ))

        btn_all = widgets.Button(description="Show all", layout=widgets.Layout(width="90px", **_FIXED))
        btn_none = widgets.Button(description="Hide all", layout=widgets.Layout(width="90px", **_FIXED))
        btn_all.on_click(lambda _: self.set_all(True))
        btn_none.on_click(lambda _: self.set_all(False))

        header = [
            widgets.HTML("<b>Papers &amp; legend</b>", layout=widgets.Layout(**_FIXED)),
            widgets.HBox([btn_all, btn_none], layout=widgets.Layout(**_FIXED)),
        ]
        self.widget = widgets.VBox(
            header + rows,
            layout=widgets.Layout(width="400px", max_height="460px", overflow="auto",
                                  border="1px solid #666", padding="6px", **_FIXED),
        )

    # ---------- wiring ----------
    def _wire(self, paper, master, expand, solo, body):
        def on_master(change):
            if self._busy:
                return
            self._busy = True
            try:
                for cb in self.curve_cbs[paper].values():
                    cb.value = change["new"]
            finally:
                self._busy = False
            self._refresh_label(paper)
            self._fire()

        def on_child(change):
            if self._busy:
                return
            self._busy = True
            try:
                master.value = all(cb.value for cb in self.curve_cbs[paper].values())
            finally:
                self._busy = False
            self._refresh_label(paper)
            self._fire()

        def on_expand(change):
            body.layout.display = "flex" if change["new"] else "none"
            expand.description = "▾" if change["new"] else "▸"

        master.observe(on_master, names="value")
        for cb in self.curve_cbs[paper].values():
            cb.observe(on_child, names="value")
        expand.observe(on_expand, names="value")
        solo.on_click(lambda _: self.solo(paper))

    def _refresh_label(self, paper):
        on = sum(cb.value for cb in self.curve_cbs[paper].values())
        self.master[paper].description = f"{paper}  ({on}/{self._counts[paper]})"

    def _fire(self):
        if self._busy:
            return
        for fn in self._callbacks:
            fn(None)

    # ---------- bulk actions ----------
    def _set_papers(self, flag_for):
        """Set every checkbox from flag_for(paper), then fire the callbacks once."""
        self._busy = True
        try:
            for paper in self.papers:
                flag = flag_for(paper)
                for cb in self.curve_cbs[paper].values():
                    cb.value = flag
                self.master[paper].value = flag
                self._refresh_label(paper)
        finally:
            self._busy = False
        self._fire()

    def set_all(self, value):
        self._set_papers(lambda paper: value)

    def solo(self, only_paper):
        self._set_papers(lambda paper: paper == only_paper)

    # ---------- public API ----------
    def on_change(self, fn):
        self._callbacks.append(fn)

    def active_labels(self):
        return {lab for p in self.papers for lab, cb in self.curve_cbs[p].items() if cb.value}

    def active_papers(self):
        return [p for p in self.papers if any(cb.value for cb in self.curve_cbs[p].values())]

    def filter(self, d, labels=None):
        """Rows of d whose curve is ticked. Pass `labels` to reuse one active_labels() call."""
        return d[d["label"].isin(self.active_labels() if labels is None else labels)]


def build_legend_selector(frames, palette_dict, marker_dict):
    return LegendSelector(frames, palette_dict, marker_dict)

In [42]:
#@title Block 3: Import CSVs and Rebuild Plot Data {display-mode: "form"}
import os, re, json, hashlib, shutil, tempfile, uuid
from pathlib import Path
import numpy as np
import pandas as pd

if not {'historic_raw_list','historic_df_list','condition_columns'} <= set(globals()):
    raise RuntimeError('Run Block 1 first, then Block 2, then this block.')

MASTER_DIR = globals().get('MASTER_DIR', '/content/drive/MyDrive/TurbineResearch')
root = Path(MASTER_DIR)
uploads = root / 'undergrad_uploads'
archive = uploads / 'archive'
archive.mkdir(parents=True, exist_ok=True)
raw_file = root / 'undergrad_raw_master.parquet'
peak_file = root / 'undergrad_peak_master.parquet'

# Column rules live in Block 2 (REQUIRED_COLS, NON_CONDITION_COLS, condition_columns).
# Required in every CSV: TSR, Cp, cToR, Re_D (header "Re" also accepted).
# Optional: N and ANY other column - kept as-is, splits curves, shows up in the curve name.

def _is_text(series):
    return pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)

def read_curve(path):
    d = pd.read_csv(path, encoding='utf-8-sig')
    d.columns = [str(c).strip() for c in d.columns]
    if 'Re' in d.columns and 'Re_D' not in d.columns:
        d = d.rename(columns={'Re': 'Re_D'})
    if d.empty:
        raise ValueError('CSV has no data rows.')
    for col in d.columns:
        if _is_text(d[col]):
            d[col] = d[col].map(lambda v: v.strip() if isinstance(v,str) else v)
            d[col] = d[col].replace('', np.nan)
    # N / cToR can also come from the filename, e.g. "Smith (N=3, cToR=0.12).csv"
    for col, pattern in {'N':r'\bN\s*=\s*(\d+)', 'cToR':r'\bcToR\s*=\s*([\d.]+)'}.items():
        match = re.search(pattern, path.stem)
        if match:
            if col not in d: d[col] = np.nan
            d[col] = d[col].fillna(float(match.group(1)))
    missing = [c for c in REQUIRED_COLS if c not in d.columns]
    if missing:
        raise ValueError(f'Missing required column(s): {", ".join(missing)}. '
                         'Every CSV needs the headers TSR, Cp, cToR and Re_D (or Re).')

    # Required + N: strict numbers (text units are rejected, not silently turned into NaN).
    strict = REQUIRED_COLS + (['N'] if 'N' in d else [])
    for col in strict:
        d[col] = pd.to_numeric(d[col], errors='raise')
    # Every other optional column: numeric if it parses cleanly, otherwise kept as text.
    for col in condition_columns(d.columns):
        if col not in strict and _is_text(d[col]):
            try: d[col] = pd.to_numeric(d[col], errors='raise')
            except (ValueError, TypeError): pass
    for col in d.select_dtypes('number'):
        if np.isinf(d[col]).any(): raise ValueError(f'{col} contains infinity.')

    if d[REQUIRED_COLS].isna().any().any():
        raise ValueError('TSR, Cp, cToR and Re_D must be present on every row; fill constants down.')
    if (d.cToR <= 0).any():
        raise ValueError('cToR must be positive.')
    if (d.Re_D <= 0).any():
        raise ValueError('Re_D must be positive.')
    if 'N' not in d: d['N'] = np.nan
    n = d['N'].dropna()
    if ((n <= 0) | (n % 1 != 0)).any():
        raise ValueError('N, when given, must be a positive integer.')
    if (d.TSR < 0).any():
        raise ValueError('Negative TSR found. Review the point before importing.')
    paper = path.stem.split('(')[0].split('[')[0].strip()
    if not paper: raise ValueError('Filename must start with the source name.')
    d['dataset_source'] = paper
    d['source_file'] = path.name
    d['ingest_file_key'] = path.name  # Reimporting this filename replaces its own rows.

    # A curve = one unique combination of ALL condition columns this file supplies.
    cols = condition_columns(d.columns)
    def identify(row):
        values = {c: None if pd.isna(row[c]) else str(row[c]) for c in cols}
        token = hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()[:16]
        return path.stem + ' | ' + token
    # Hash once per UNIQUE set of conditions instead of once per row.
    group_id = d.groupby(cols, sort=False, dropna=False).ngroup()
    first = ~group_id.duplicated()
    tokens = pd.Series(d[first].apply(identify, axis=1).to_numpy(), index=group_id[first].to_numpy())
    d['config_id'] = group_id.map(tokens)
    d['label'] = d['config_id']
    # This project's explicit circumference-based solidity convention (needs N).
    d['solidity'] = d.N * d.cToR / (2*np.pi)
    d['Dynamic_Flow_Opacity'] = d.solidity * d.TSR
    d['Dynamic_Blockage'] = d.Dynamic_Flow_Opacity
    if d.N.isna().any(): print('    N not supplied for some rows: flow opacity is unavailable for them '
                               '(they still appear in TSR plots).', flush=True)
    if (d.Cp > .593).any(): print('    NOTE: Cp > 0.593; verify normalization/calibration.', flush=True)
    return d

def normalize_types(d):
    d = d.copy()
    for c in d.select_dtypes(include=['object','string']).columns:
        d[c] = d[c].map(lambda v: None if pd.isna(v) else str(v))
    return d

def stage_parquet(d, target):
    fd, temporary = tempfile.mkstemp(prefix=target.stem+'_', suffix='.parquet', dir=root)
    os.close(fd)
    try: normalize_types(d).to_parquet(temporary, index=False)
    except Exception:
        os.unlink(temporary); raise
    return temporary

def combine(frames):
    frames = [d for d in frames if not d.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

print('Loading saved undergraduate data...', flush=True)
df_raw_cum = pd.read_parquet(raw_file) if raw_file.exists() else pd.DataFrame()
files = sorted(uploads.glob('*.csv'), key=lambda p:p.name.casefold())
print(f'Found {len(files)} CSV file(s) in {uploads}', flush=True)
accepted = []; failures = []; new_frames = {}
for number, path in enumerate(files,1):
    print(f'[{number}/{len(files)}] Reading {path.name}', flush=True)
    try:
        d = read_curve(path)
        new_frames[path.name] = d
        accepted.append(path)
        print(f'    Validated {len(d)} points / {d.config_id.nunique()} curve(s).', flush=True)
    except Exception as exc:
        failures.append(path.name)
        print(f'    ERROR: {exc}\n    Left in uploads for correction.', flush=True)

# Merge ONCE at the end (the old loop re-concatenated the whole table for every file).
# Reimporting a filename replaces its own rows.
if new_frames and 'ingest_file_key' in df_raw_cum:
    df_raw_cum = df_raw_cum[~df_raw_cum.ingest_file_key.isin(list(new_frames))]
df_raw_cum = combine([df_raw_cum, *new_frames.values()])

if not df_raw_cum.empty:
    if 'config_id' not in df_raw_cum or df_raw_cum.config_id.isna().any():
        raise ValueError('Saved undergraduate master has missing config_id values. No files archived; inspect the old master.')
    df_raw_cum = df_raw_cum.reset_index(drop=True)
    df_peak_cum = df_raw_cum.loc[df_raw_cum.groupby('config_id',sort=False).Cp.idxmax()].copy()
else:
    df_peak_cum = pd.DataFrame()

df_raw = combine(historic_raw_list + [df_raw_cum])
df_peaks = combine(historic_df_list + [df_peak_cum])
if df_raw.empty:
    print('No valid data available. Nothing saved or archived.', flush=True)
else:
    print('Saving undergraduate data and rebuilding BOTH plot master files...', flush=True)
    targets = [(df_raw_cum,raw_file),(df_peak_cum,peak_file),
               (df_raw,root/'master_raw_matrix.parquet'),
               (df_peaks,root/'master_peak_matrix.parquet')]
    staged = []
    try:
        # Finish all serialization before replacing any master. Upload files remain
        # available if saving fails; a rerun replaces each accepted file idempotently.
        for data,target in targets: staged.append((stage_parquet(data,target),target))
        for temporary,target in staged: os.replace(temporary,target)
    finally:
        for temporary,_ in staged:
            if os.path.exists(temporary): os.unlink(temporary)
    for path in accepted:
        try:
            destination = archive / path.name
            if destination.exists():
                backup_dir = archive / 'previous' / uuid.uuid4().hex
                backup_dir.mkdir(parents=True)
                shutil.move(str(destination),str(backup_dir/destination.name))
            shutil.move(str(path),str(destination))
            print(f'    Archived: {path.name}', flush=True)
        except Exception as exc:
            print(f'    Archive warning for {path.name}: {exc}. Data saved; rerunning is safe.',flush=True)
    print(f'DONE: {len(accepted)} accepted; {len(failures)} failed. '
          f'Master: {len(df_raw)} points / {len(df_peaks)} peaks.',flush=True)
    print('Now rerun Action 1 and Action 2 to refresh their widgets and plots.',flush=True)

Loading saved undergraduate data...
Found 0 CSV file(s) in /content/drive/MyDrive/TurbineResearch/undergrad_uploads
Saving undergraduate data and rebuilding BOTH plot master files...
DONE: 0 accepted; 0 failed. Master: 5160 points / 289 peaks.
Now rerun Action 1 and Action 2 to refresh their widgets and plots.


In [43]:
#@title Action 1: Interactive Domain Tessellation (TSR vs. Opacity) {display-mode: "form"}
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import ipywidgets as widgets
from IPython.display import display, clear_output


def show_tessellation(master_dir="/content/drive/MyDrive/TurbineResearch"):
    """
    Everything lives INSIDE this function on purpose. The old version kept `out_plot`,
    `legend`, `df`, ... as notebook-level globals, so running Action 2 afterwards
    overwrote them and Action 1's widgets started drawing into Action 2's output.
    Now each action has its own private copies and they can't touch each other.
    """
    raw_path = os.path.join(master_dir, "master_raw_matrix.parquet")
    if not os.path.exists(raw_path):
        print("Missing master_raw_matrix.parquet. Run Block 3 first.")
        return None

    # ----------------------------------------------------------
    # 1. LOAD + SHARED CLEANUP
    # ----------------------------------------------------------
    df = prepare_frames([pd.read_parquet(raw_path)])[0]
    if "Dynamic_Flow_Opacity" not in df.columns:
        raise ValueError("Missing Dynamic_Flow_Opacity. Run the ingestion and compilation cells first.")

    num_cols = ["TSR", "Dynamic_Flow_Opacity", "Re_D"]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

    valid = np.isfinite(df[["TSR", "Dynamic_Flow_Opacity"]].to_numpy(dtype=float)).all(axis=1)
    excluded_count = int((~valid).sum())
    df = df.loc[valid]
    if excluded_count:
        print(f"Excluded {excluded_count} rows with missing or invalid TSR/opacity coordinates from this view.")
    if df.empty:
        print("No valid TSR/opacity points are available.")
        return None

    # ----------------------------------------------------------
    # 2. FIXED AXIS LIMITS
    # ----------------------------------------------------------
    def padded_limits(series):
        low, high = float(series.min()), float(series.max())
        span = high - low
        padding = span * 0.05 if span > 0 else max(abs(low) * 0.05, 0.1)
        return low - padding, high + padding

    xlim = padded_limits(df["TSR"])
    ylim = padded_limits(df["Dynamic_Flow_Opacity"])
    extent = [xlim[0], xlim[1], ylim[0], ylim[1]]

    valid_res = df.loc[np.isfinite(df["Re_D"]) & (df["Re_D"] > 0), "Re_D"]
    if valid_res.empty:
        re_min, re_max = 0.0, 10.0
    else:
        re_min = float(np.floor(valid_res.min() / 1e4) / 10.0)
        re_max = float(np.ceil(valid_res.max() / 1e4) / 10.0)
        if re_max <= re_min:
            re_max = re_min + 0.1

    # ----------------------------------------------------------
    # 3. CONTROLS
    # ----------------------------------------------------------
    palette_dict, marker_dict = make_style_dicts(sorted(df["dataset_source"].unique()))
    legend = build_legend_selector([df], palette_dict, marker_dict)

    view_mode = widgets.RadioButtons(
        options=[
            "Combined Tessellation (Re_D Masked)",
            "Combined Tessellation (Unmasked All Data)",
            "Dataset Breakdown View (Re_D Masked)",
            "Dataset Breakdown View (Unmasked All Data)",
        ],
        value="Combined Tessellation (Unmasked All Data)",
        description="Layout:",
        layout=widgets.Layout(width="max-content"),
    )
    red_slider = widgets.FloatRangeSlider(
        value=[re_min, re_max], min=re_min, max=re_max, step=0.1,
        description="Re_D (×10⁵):", readout_format=".1f", continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="420px", display="none"),
    )
    out_plot = widgets.Output()

    HEX = dict(gridsize=30, extent=extent, mincnt=1)

    def combined_max_count(data):
        """Largest hexagon count over ALL selected points (shared colour scale).
        Uses a throwaway Figure that never touches pyplot's global state."""
        ax = Figure().subplots()
        counts = ax.hexbin(data["TSR"], data["Dynamic_Flow_Opacity"], **HEX).get_array()
        return float(counts.max()) if len(counts) else 1.0

    # ----------------------------------------------------------
    # 4. PLOTTING
    # ----------------------------------------------------------
    def update(*_):
        is_unmasked = "Unmasked" in view_mode.value
        is_combined = view_mode.value.startswith("Combined")
        red_slider.layout.display = "none" if is_unmasked else "flex"

        with out_plot:
            clear_output(wait=True)

            selected = legend.filter(df)
            if selected.empty:
                print("Nothing selected. Tick a paper (or expand it and pick curves) in the legend panel.")
                return

            if is_unmasked:
                data = selected
            else:
                lo, hi = red_slider.value
                data = selected[np.isfinite(selected["Re_D"]) & (selected["Re_D"] > 0)
                                & selected["Re_D"].between(lo * 1e5, hi * 1e5)]
                removed = len(selected) - len(data)
                if removed:
                    print(f"Re_D filter excludes {removed} points with missing, invalid, or out-of-range Re_D.")

            if data.empty:
                print("No points match this selection. Choose Unmasked All Data or adjust the Re_D range.")
                return

            if is_combined:
                title = "All selected papers"
                if not is_unmasked:
                    title += f"\nRe_D: {lo:g}–{hi:g} × 10⁵"
                plot_data = [(title, data)]
                cols, rows, figsize = 1, 1, (7, 6.5)
            else:
                present = set(data["dataset_source"])
                plot_data = [(p, data[data["dataset_source"] == p]) for p in legend.papers if p in present]
                cols = min(3, len(plot_data))
                rows = int(np.ceil(len(plot_data) / cols))
                figsize = (5.5 * cols, 5.2 * rows)

            fig = None
            try:
                fig, axes = plt.subplots(rows, cols, figsize=figsize, sharex=True, sharey=True,
                                         squeeze=False, constrained_layout=True)
                axes_list = axes.ravel()
                hexes = []

                for ax, (source, subset) in zip(axes_list, plot_data):
                    hexes.append(ax.hexbin(
                        subset["TSR"], subset["Dynamic_Flow_Opacity"], vmin=0,
                        cmap="plasma", edgecolors="white", linewidths=0.5, **HEX))
                    ax.set_title(source, fontsize=12, fontweight="bold")
                    ax.set_xlim(xlim)
                    ax.set_ylim(ylim)
                    ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
                    ax.axvline(0, color="gray", linewidth=0.7, linestyle="--")
                    ax.set_xlabel(r"Tip Speed Ratio ($\lambda$)")
                    ax.set_ylabel(r"Dynamic Flow Opacity ($\phi = \sigma\lambda$)")
                    ax.grid(True, linestyle=":", alpha=0.4)
                    ax.set_axisbelow(True)

                for ax in axes_list[len(plot_data):]:
                    ax.set_visible(False)

                # Shared colour scale. One panel: its own max. Several: the combined max
                # (computed only when needed instead of always drawing a dummy figure).
                if len(plot_data) == 1:
                    vmax = max(1.0, float(hexes[0].get_array().max()))
                else:
                    vmax = max(1.0, combined_max_count(data))
                for hb in hexes:
                    hb.set_clim(0, vmax)

                cbar = fig.colorbar(hexes[-1], ax=list(axes_list[:len(plot_data)]), fraction=0.035, pad=0.03)
                cbar.set_label("Digitized points per hexagon")
                fig.suptitle("Domain Tessellation — TSR vs. Flow Opacity", fontsize=15, fontweight="bold")
                plt.show()
            finally:
                if fig is not None:
                    plt.close(fig)

            print(f"Displaying {len(data):,} points from {data['dataset_source'].nunique()} papers "
                  f"({data['label'].nunique()} curves).")

    # ----------------------------------------------------------
    # 5. DISPLAY
    # ----------------------------------------------------------
    legend.on_change(update)
    view_mode.observe(update, names="value")
    red_slider.observe(update, names="value")

    controls = widgets.HBox(
        [widgets.VBox([view_mode, red_slider]), legend.widget],
        layout=widgets.Layout(align_items="flex-start"),
    )
    display(controls, out_plot)
    update()
    return dict(controls=controls, out=out_plot, legend=legend, view_mode=view_mode,
                red_slider=red_slider, update=update)


_tessellation_ui = show_tessellation()   # (assigned so the dict isn't echoed under the cell)

📚 7 papers detected (from 7 raw dataset names).


Output()

In [44]:
#@title Action 2: 10-Phase Experimental Data LOESS Storyboard {display-mode: "form"}
import os
import re
import textwrap
import traceback
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from statsmodels.nonparametric.smoothers_lowess import lowess

X_COLS = ("TSR", "Dynamic_Flow_Opacity")
LABEL_TSR = r"Tip Speed Ratio ($\lambda$)"
LABEL_OP = r"Dynamic Flow Opacity ($\phi = \sigma \cdot \lambda$)"
X_LABELS = (LABEL_TSR, LABEL_OP)
PHASES = [
    ("Sample Configs (Raw Sweeps)", 1),
    ("Sample Configs (Peak Isolated)", 2),
    ("Full Dataset (All Scatter Points)", 3),
    ("Full Dataset (All Sweeps)", 4),
    ("Full Dataset (Global LOESS Trends)", 5),
    ("Full Dataset (LOESS Breakdown by Blade Count)", 6),
    ("Peak Performance (Global Swarm & Count)", 7),
    ("Peak Performance (Global Polynomial Trends)", 8),
    ("Peak Performance (Architectural LOESS Breakdown)", 9),
    ("Peak Performance (Common Axis Overlay)", 10),
]


def calc_poly_rmse(x, y, deg=2):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) <= deg or len(np.unique(x)) <= deg:
        return np.nan
    poly = np.poly1d(np.polyfit(x, y, deg))
    return float(np.sqrt(np.mean((y - poly(x)) ** 2)))


def loess_line(ax, data, xcol, **line_kws):
    """
    Same LOESS curve sns.regplot(lowess=True) draws, but with statsmodels' `delta`
    shortcut (skips fitting between points closer than 1% of the x-range). On big
    datasets this is far faster and visually indistinguishable.
    """
    x = data[xcol].to_numpy(dtype=float)
    y = data["Cp"].to_numpy(dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 3:
        return
    span = x.max() - x.min()
    fit = lowess(y, x, frac=2 / 3, it=3, delta=0.01 * span if span > 0 else 0.0)
    ax.plot(fit[:, 0], fit[:, 1], **line_kws)


def show_storyboard(master_dir="/content/drive/MyDrive/TurbineResearch"):
    """
    All state is local to this function (see note in Action 1): the storyboard's
    widgets can no longer be redirected by running another action cell.
    """
    raw_path = os.path.join(master_dir, "master_raw_matrix.parquet")
    peak_path = os.path.join(master_dir, "master_peak_matrix.parquet")
    if not os.path.exists(raw_path) or not os.path.exists(peak_path):
        print("❌ ERROR: Missing master matrices. Ensure Block 3 exported both files.")
        return None

    df_raw, df_peaks = prepare_frames([pd.read_parquet(raw_path), pd.read_parquet(peak_path)])

    available = sorted(set(df_raw["dataset_source"]) | set(df_peaks["dataset_source"]))
    palette_dict, marker_dict = make_style_dicts(available)

    # Axis limits only depend on the full dataset, so compute them once (not per redraw).
    def padded(series, pad=0.125):
        lo, hi = series.min(), series.max()
        return (lo - pad * (hi - lo), hi + pad * (hi - lo))

    X_LIMS = (padded(df_raw["TSR"]), padded(df_raw["Dynamic_Flow_Opacity"]))
    VIRIDIS4 = sns.color_palette("viridis", 4)

    # ------------------------------------------------------------------
    # UI
    # ------------------------------------------------------------------
    out_plot = widgets.Output()
    phase_selector = widgets.Dropdown(options=PHASES, value=1, description="Phase:",
                                      layout={"width": "450px"})
    outlier_toggle = widgets.Checkbox(value=False, description="Remove Outliers (Z > 2.5)")
    negative_toggle = widgets.Checkbox(value=False, description="Remove Negative Cp")
    blind_toggle = widgets.Checkbox(value=False, description="Blind Datasets (Phase 5)")
    legend = build_legend_selector([df_raw, df_peaks], palette_dict, marker_dict)
    phase_titles = {v: k for k, v in PHASES}

    # ------------------------------------------------------------------
    # small plotting helpers
    # ------------------------------------------------------------------
    def style_scatter(ax, data, xcol, **kw):
        sns.scatterplot(data=data, x=xcol, y="Cp", style="dataset_source", hue="dataset_source",
                        markers=marker_dict, palette=palette_dict, ax=ax, **kw)

    def grid_axes(axes, i, j, y_lim):
        ax = axes[i, j]
        ax.set_ylim(y_lim)
        ax.set_xlim(X_LIMS[j])
        ax.grid(True, linestyle="--", alpha=0.5)
        if i == 3:
            ax.set_xlabel(X_LABELS[j])
        return ax

    # ------------------------------------------------------------------
    # main redraw
    # ------------------------------------------------------------------
    def update(*_):
        step = phase_selector.value
        labels = legend.active_labels()           # computed ONCE per redraw
        act_raw = legend.filter(df_raw, labels)
        act_pks = legend.filter(df_peaks, labels)

        d_full, d_pks = act_raw, act_pks
        if negative_toggle.value:
            d_full = d_full[d_full["Cp"] >= 0]
            d_pks = d_pks[d_pks["Cp"] >= 0]
        if outlier_toggle.value and not d_pks.empty:
            std = d_pks["Cp"].std()
            if std > 0:                            # (std 0 / one point -> nothing to flag)
                z = ((d_pks["Cp"] - d_pks["Cp"].mean()) / std).abs()
                keep = set(d_pks.loc[z < 2.5, "label"])
                d_full = d_full[d_full["label"].isin(keep)]
                d_pks = d_pks[d_pks["label"].isin(keep)]

        with out_plot:
            clear_output(wait=True)
            display(HTML(f"<h3>Phase {step}: {phase_titles[step]}</h3>"))

            if not labels:
                print("⚠️ Nothing selected. Tick a paper (or expand it and pick curves) in the legend panel.")
                return

            y_ref = d_full["Cp"] if step <= 6 else d_pks["Cp"]
            if len(y_ref) == 0:
                print("No data available with the current filters.")
                return

            cp_range = max(0.1, y_ref.max() - y_ref.min())
            Y_LIM = (y_ref.min() - 0.05 * cp_range, y_ref.max() + 0.05 * cp_range)

            fig = None
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")

                    if step in (6, 9):
                        fig, axes = plt.subplots(4, 2, figsize=(17, 18))
                        pairs = None
                    else:
                        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(19, 6.5))
                        pairs = list(zip((ax1, ax2), X_COLS))

                    # --- RAW SWEEP PHASES (1-6) ---------------------------------
                    if step in (1, 2):
                        # 5 reproducible random curves out of whatever is currently ticked
                        avail = sorted(act_raw["label"].unique())
                        rng = np.random.RandomState(42)
                        picked = rng.choice(avail, min(5, len(avail)), replace=False) if avail else []
                        samp_raw = act_raw[act_raw["label"].isin(picked)]
                        samp_pks = act_pks[act_pks["label"].isin(picked)]

                        if step == 1:
                            for ax, x in pairs:
                                sns.lineplot(data=samp_raw, x=x, y="Cp", hue="label", style="dataset_source",
                                             markers=marker_dict, dashes=False, alpha=0.6, ax=ax)
                        else:
                            if negative_toggle.value:
                                samp_pks = samp_pks[samp_pks["Cp"] >= 0]
                            for ax, x in pairs:
                                sns.lineplot(data=samp_raw, x=x, y="Cp", hue="label", legend=False, alpha=0.3, ax=ax)
                                style_scatter(ax, samp_pks, x, s=140, zorder=5, edgecolor="black")

                    elif step == 3:
                        for ax, x in pairs:
                            style_scatter(ax, d_full, x, alpha=0.4)

                    elif step == 4:
                        multi = d_full[d_full.groupby("label")["label"].transform("size") > 1]
                        for ax, x in pairs:
                            if multi.empty:
                                ax.text(0.5, 0.5, "No multi-point sweeps available.",
                                        ha="center", va="center", transform=ax.transAxes)
                            else:
                                sns.lineplot(data=multi, x=x, y="Cp", units="label", estimator=None,
                                             alpha=0.1, ax=ax, color="blue")

                    elif step == 5:
                        for ax, x in pairs:
                            if blind_toggle.value:
                                sns.scatterplot(data=d_full, x=x, y="Cp", color="gray", alpha=0.4, ax=ax)
                            else:
                                style_scatter(ax, d_full, x, alpha=0.2)
                            loess_line(ax, d_full, x, color="red", lw=3)

                    elif step == 6:
                        for i, n in enumerate((1, 2, 3, 4)):
                            subset = d_full[d_full["N"] == n]
                            for j, x in enumerate(X_COLS):
                                ax = grid_axes(axes, i, j, Y_LIM)
                                sns.scatterplot(data=d_full, x=x, y="Cp", color="lightgray", alpha=0.1,
                                                ax=ax, edgecolor=None)
                                style_scatter(ax, subset, x, alpha=0.4)
                                if len(subset) > 2:
                                    loess_line(ax, subset, x, color="darkred", lw=3)
                                rmse = calc_poly_rmse(subset[x], subset["Cp"])
                                ax.set_title(f"N={n}: Full Data (Poly RMSE: {rmse:.3f})", fontweight="bold")

                    # --- PEAK DATA PHASES (7-10) --------------------------------
                    elif step == 7:
                        for ax, x in pairs:
                            sns.scatterplot(data=d_pks, x=x, y="Cp", hue="N", style="dataset_source",
                                            markers=marker_dict, palette="viridis", s=80, alpha=0.8,
                                            edgecolor="k", ax=ax)
                            ax.set_title(f"Peak Swarm (Configs mapped: {len(d_pks)})", fontweight="bold")

                    elif step == 8:
                        for ax, x in pairs:
                            style_scatter(ax, d_pks, x, alpha=0.6)
                            sns.regplot(data=d_pks, x=x, y="Cp", scatter=False, order=2, color="black",
                                        line_kws={"lw": 2}, ax=ax)
                            rmse = calc_poly_rmse(d_pks[x], d_pks["Cp"])
                            ax.set_title(f"Peak Performance (Poly RMSE: {rmse:.3f})", fontweight="bold")

                    elif step == 9:
                        for i, n in enumerate((1, 2, 3, 4)):
                            subset = d_pks[d_pks["N"] == n]
                            for j, x in enumerate(X_COLS):
                                ax = grid_axes(axes, i, j, Y_LIM)
                                sns.regplot(data=d_pks, x=x, y="Cp", scatter=False, color="gray", ax=ax,
                                            line_kws={"linestyle": "--", "alpha": 0.3}, order=2, ci=None)
                                sns.scatterplot(data=d_pks, x=x, y="Cp", color="lightgray", alpha=0.3,
                                                ax=ax, edgecolor=None)
                                style_scatter(ax, subset, x, s=80, edgecolor="k", legend=False)
                                if len(subset) > 2:
                                    loess_line(ax, subset, x, color="darkred", lw=2)
                                rmse = calc_poly_rmse(subset[x], subset["Cp"])
                                ax.set_title(f"N={n}: Peaks (Poly RMSE: {rmse:.3f})", fontweight="bold")

                    elif step == 10:
                        for i, n in enumerate((1, 2, 3, 4)):
                            subset = d_pks[d_pks["N"] == n]
                            if subset.empty:
                                continue
                            for ax, x in pairs:
                                style_scatter(ax, subset, x, alpha=0.6, s=70, edgecolor="k")
                                if len(subset) > 2:
                                    loess_line(ax, subset, x, color=VIRIDIS4[i], lw=2, label=f"N={n}")
                        ax1.set_title("Common Axis: Peak Cp vs. TSR", fontweight="bold")
                        ax2.set_title("Common Axis: Peak Cp vs. Opacity", fontweight="bold")

                    # --- GLOBAL FORMATTING (single-row figures) -----------------
                    if pairs is not None:
                        ax1.set_xlabel(LABEL_TSR)
                        ax1.set_ylabel(r"Power Coefficient ($C_P$)")
                        ax2.set_xlabel(LABEL_OP)
                        ax2.set_ylabel("")
                        for ax, j in ((ax1, 0), (ax2, 1)):
                            ax.set_ylim(Y_LIM)
                            ax.set_xlim(X_LIMS[j])
                            ax.grid(True, linestyle=":", alpha=0.6)
                        if step < 7:
                            ax1.set_title(ax1.get_title() or "Full Dataset: TSR vs Efficiency", fontweight="bold")
                            ax2.set_title(ax2.get_title() or "Full Dataset: Flow Opacity vs Efficiency", fontweight="bold")

                    # --- SIDE LEGEND: one tidy column to the right of the plots --
                    handles, lbls = [], []
                    for ax in fig.axes:
                        h, l = ax.get_legend_handles_labels()
                        handles.extend(h)
                        lbls.extend(l)
                        if ax.get_legend() is not None:
                            ax.get_legend().remove()

                    by_label = dict(zip(lbls, handles))
                    dict_N, dict_papers, dict_curves = {}, {}, {}
                    for l, h in by_label.items():
                        if l in ("N", "dataset_source", "label"):
                            continue
                        if str(l) in ("1", "2", "3", "4", "1.0", "2.0", "3.0", "4.0"):
                            dict_N[f"N={int(float(l))}"] = h
                        elif re.fullmatch(r"N=\d+", str(l)):
                            dict_N[str(l)] = h
                        elif l in palette_dict:
                            dict_papers[l] = h
                        else:
                            dict_curves[l] = h

                    entry_handles, entry_labels = [], []
                    for title, dct in (("Blade count", dict_N), ("Papers", dict_papers), ("Curves", dict_curves)):
                        if dct:
                            entry_handles.append(Line2D([], [], linestyle="none", marker="None"))
                            entry_labels.append(f"— {title} —")
                            for l, h in dct.items():
                                entry_handles.append(h)
                                entry_labels.append(textwrap.fill(str(l), 36))

                    n_lines = sum(l.count("\n") + 1 for l in entry_labels)
                    w_in, h_in = fig.get_size_inches()
                    fig.set_size_inches(w_in, max(h_in, 0.24 * n_lines + 1.2))

                    fig.tight_layout(rect=[0, 0, 0.79, 1])
                    if entry_handles:
                        fig.legend(entry_handles, entry_labels, loc="upper left",
                                   bbox_to_anchor=(0.80, 0.985), ncol=1, frameon=True,
                                   fontsize=9, borderaxespad=0.0, labelspacing=0.5)

                    plt.show()

            except Exception as e:
                clear_output(wait=False)
                print(f"❌ Plotting Execution Error in Phase {step}: {e}")
                traceback.print_exc()
            finally:
                if fig is not None:
                    plt.close(fig)

    phase_selector.observe(update, names="value")
    outlier_toggle.observe(update, names="value")
    negative_toggle.observe(update, names="value")
    blind_toggle.observe(update, names="value")
    legend.on_change(update)

    controls = widgets.HBox(
        [widgets.VBox([phase_selector, outlier_toggle, negative_toggle, blind_toggle]), legend.widget],
        layout=widgets.Layout(align_items="flex-start"),
    )
    display(controls, out_plot)
    update()
    return dict(controls=controls, out=out_plot, legend=legend, phase=phase_selector,
                outliers=outlier_toggle, negatives=negative_toggle, blind=blind_toggle, update=update)


_storyboard_ui = show_storyboard()   # (assigned so the dict isn't echoed under the cell)

📚 7 papers detected (from 7 raw dataset names).


Output()

In [45]:
#@title Action 3: Dataset Manager {display-mode: "form"}
import os
import shutil
import tempfile
from pathlib import Path
from datetime import datetime
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Disable any old deletion widget when this cell is rerun.
for control in globals().get("_delete_controls", []):
    control.disabled = True


def make_delete_widget():
    root = Path("/content/drive/MyDrive/TurbineResearch")
    names = [
        "undergrad_raw_master.parquet",
        "undergrad_peak_master.parquet",
        "master_raw_matrix.parquet",
        "master_peak_matrix.parquet",
    ]

    paper = widgets.Dropdown(
        description="Paper:", options=[("Choose paper", None)],
        layout=widgets.Layout(width="95%")
    )
    curve = widgets.Dropdown(
        description="Curve:", options=[("Choose paper first", None)],
        layout=widgets.Layout(width="95%")
    )
    refresh = widgets.Button(description="Refresh list")
    preview = widgets.Button(description="Preview deletion")
    confirm = widgets.Checkbox(
        description="I checked the preview and want to delete this data.",
        value=False, disabled=True, indent=False,
        layout=widgets.Layout(width="95%")
    )
    delete = widgets.Button(
        description="Delete selected", button_style="danger", disabled=True
    )
    output = widgets.Output()
    state = {"raw": pd.DataFrame(), "pending": None}

    def reset(*_):
        state["pending"] = None
        confirm.value = False
        confirm.disabled = True
        delete.disabled = True

    def stamp(path):
        if not path.exists():
            return None
        stat = path.stat()
        return stat.st_size, stat.st_mtime_ns

    def choose_paper(*_):
        reset()
        raw = state["raw"]
        if paper.value is None:
            curve.options = [("Choose paper first", None)]
            return
        rows = raw[raw["dataset_source"].astype(str).eq(paper.value)]
        ids = sorted(rows["config_id"].dropna().astype(str).unique())
        curve.options = (
        [("All curves in this paper", "__ALL__")]
        + [(value, value) for value in ids]
        )

    def reload(*_):
        reset()
        raw_path = root / names[0]
        raw = pd.read_parquet(raw_path) if raw_path.exists() else pd.DataFrame()
        if not raw.empty:
            required = {"dataset_source", "config_id"}
            if not required.issubset(raw.columns):
                raise ValueError("Imported master is missing dataset_source or config_id.")
        state["raw"] = raw
        sources = (
            sorted(raw["dataset_source"].dropna().astype(str).unique())
            if not raw.empty else []
        )
        paper.options = [("Choose paper", None)] + [(s, s) for s in sources]
        paper.value = None
        choose_paper()

    def show_preview(_):
        reset()
        with output:
            clear_output(wait=True)
            if paper.value is None or curve.value is None:
                print("Choose a paper first.")
                return
            try:
                raw = pd.read_parquet(root / names[0])
                selected = raw["dataset_source"].astype(str).eq(paper.value)
                if curve.value != "__ALL__":
                    selected &= raw["config_id"].astype(str).eq(curve.value)
                rows = raw.loc[selected]
                ids = set(rows["config_id"].dropna().astype(str))
                if not ids:
                    print("No matching data. Click Refresh list.")
                    return

                tables, stamps = {}, {}
                for name in names:
                    path = root / name
                    stamps[name] = stamp(path)
                    if not path.exists():
                        continue
                    data = pd.read_parquet(path)
                    if not data.empty and "config_id" not in data:
                        raise ValueError(f"{name} has no config_id.")
                    mask = (
                        data["config_id"].astype(str).isin(ids)
                        if "config_id" in data
                        else pd.Series(False, index=data.index)
                    )
                    tables[name] = data.loc[~mask].copy()
                    print(f"{name}: {mask.sum()} rows to delete")

                # Show every condition this data actually has (no hard-coded column list).
                skip = {"TSR", "Cp", "solidity", "Dynamic_Flow_Opacity", "Dynamic_Blockage",
                        "dataset_source", "original_dataset_source", "ingest_file_key", "label"}
                columns = [
                    c for c in rows.columns
                    if c not in skip and not str(c).startswith(("Unnamed:", "assumed_"))
                    and rows[c].notna().any()
                ]
                display(rows[columns].drop_duplicates())
                state["pending"] = (tables, stamps)
                confirm.disabled = False
            except Exception as exc:
                print(f"Preview stopped: {exc}")

    def enable_delete(*_):
        delete.disabled = not (
            confirm.value and state["pending"] is not None
        )

    def delete_selected(_):
        if not confirm.value or state["pending"] is None:
            return

        tables, stamps = state["pending"]
        reset()  # Consume confirmation so another click cannot repeat deletion.
        staged, replaced = {}, []
        backup = None

        with output:
            clear_output(wait=True)
            try:
                if any(stamp(root / n) != s for n, s in stamps.items()):
                    raise RuntimeError("Data changed. Preview the selection again.")

                backup_root = root / "deletion_backups"
                backup_root.mkdir(parents=True, exist_ok=True)
                backup = Path(tempfile.mkdtemp(
                    prefix=datetime.now().strftime("%Y%m%d_%H%M%S_"),
                    dir=backup_root
                ))

                # Back up and prepare every table before replacing any.
                for name, data in tables.items():
                    shutil.copy2(root / name, backup / name)
                    fd, temporary = tempfile.mkstemp(suffix=".parquet", dir=root)
                    os.close(fd)
                    staged[name] = temporary
                    data.to_parquet(temporary, index=False)

                for name, temporary in staged.items():
                    os.replace(temporary, root / name)
                    replaced.append(name)

            except Exception as exc:
                for name in replaced:
                    shutil.copy2(backup / name, root / name)
                print(f"Deletion stopped: {exc}")
                if replaced:
                    print("Changed tables restored from backup.")
                return
            finally:
                for temporary in staged.values():
                    if os.path.exists(temporary):
                        os.unlink(temporary)

            reload()
            print("Selected data deleted. Archived CSVs are retained.")
            print(f"Backup saved in: {backup}")
            print("Rerun Action 1 and Action 2 to refresh your graphs.")

    paper.observe(choose_paper, names="value")
    curve.observe(reset, names="value")
    confirm.observe(enable_delete, names="value")
    refresh.on_click(reload)
    preview.on_click(show_preview)
    delete.on_click(delete_selected)

    reload()
    display(widgets.VBox([
        widgets.HTML("<b>Delete imported data</b>"),
        paper, curve,
        widgets.HBox([refresh, preview]),
        confirm, delete, output
    ]))
    return [paper, curve, refresh, preview, confirm, delete]


_delete_controls = make_delete_widget()